# Example for a Max-Ent restricted evolution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qalma.model import build_spin_chain
from qalma.operators.states import GibbsDensityOperator, GibbsProductDensityOperator
from qalma.evolution import adaptive_maxent_evolution, qutip_me_solve
from qalma.meanfield import variational_quadratic_mfa
ZEEMAN = 1
BETA = .5

# Build a spin chain with a Heisenberg antiferromagnetic coupling:
SYSTEM = build_spin_chain(6)
# Observables
SX_locals = [SYSTEM.site_operator(f"Sx@{pos}") for pos in SYSTEM.sites]
SY_locals = [SYSTEM.site_operator(f"Sy@{pos}") for pos in SYSTEM.sites]
SZ_locals = [SYSTEM.site_operator(f"Sz@{pos}") for pos in SYSTEM.sites]
SX_TOTAL = sum(SX_locals)
SZ_TOTAL = SYSTEM.global_operator("Sz")

# Hamiltonian:
H=SYSTEM.global_operator("Hamiltonian") + ZEEMAN * SZ_TOTAL

# Initial state
RHO_0 = GibbsProductDensityOperator(BETA*SX_locals[1])


TS=np.linspace(0,10,100)

In [ ]:
# Observables to be considered
obs = {"Sx_1":SX_locals[1],"Sx_0":SX_locals[0],"Sx_2":SX_locals[2],"Sx_3":SX_locals[3],
      "SY_1":SY_locals[1],"Sy_0":SY_locals[0],"Sy_2":SY_locals[2],"Sy_3":SY_locals[3],
       "Sz_1":SZ_locals[1],"Sz_0":SZ_locals[0],"Sz_2":SZ_locals[2],"Sz_3":SZ_locals[3],
      }


sp_hs = lambda x,y : np.real((x*y).tr())
# Exact (qutip) evolution
print("solving qutip evolution")
result_qutip = qutip_me_solve(H, RHO_0, TS, e_ops=obs).expect_ops
# Restricted Max-Ent evolution with a single basis built from K(0)
# (first order, no memory). States are not stored when e_ops is given.
print("solving single-basis evolution")
simulation_single = adaptive_maxent_evolution(
    H, BETA*SX_locals[1], TS, order=3, n_body=2,
    adaptive=False, memory_time=None, e_ops=obs,
)
result_single = simulation_single.expect_ops

# Adaptive Max-Ent evolution with memory closure (the recommended method)
print("solving adaptive evolution")
simulation_adaptive = adaptive_maxent_evolution(
    H, BETA*SX_locals[1], TS, order=3, n_body=2, e_ops=obs,
)
result_adaptive = simulation_adaptive.expect_ops
print("memory times:", simulation_adaptive.stats["memory times"])


In [ ]:
for key in obs:
    plt.plot(TS, result_qutip[key], label="exact")
    plt.plot(TS, result_single[key], label="single basis")
    plt.plot(TS[:len(result_adaptive[key])], result_adaptive[key], label="adaptive", ls="-.")
    plt.legend()
    plt.title(key)
    plt.show()
